In [1]:
import nest_asyncio
nest_asyncio.apply()

In [1]:
from llama_index.core.node_parser import HTMLNodeParser
from llama_index.core import SimpleDirectoryReader

# html 파일 불러오기
html_docs = SimpleDirectoryReader(input_dir='./data', required_exts=['.html']).load_data()

# html노드파서 사용
parser = HTMLNodeParser()
html_nodes = parser.get_nodes_from_documents(html_docs)

In [2]:
# html노드 개수
len(html_nodes)

4

In [3]:
# 첫 번째 노드 텍스트
html_nodes[0].text

'RAG (Retrieval Augmented Generation)'

In [4]:
# 첫 번째 노드 메타데이터
html_nodes[0].metadata

{'file_path': 'c:\\Users\\jangc\\ai_class\\llm_labs\\data\\html_example.html',
 'file_name': 'html_example.html',
 'file_type': 'text/html',
 'file_size': 458,
 'creation_date': '2026-05-28',
 'last_modified_date': '2026-06-03',
 'tag': 'h1'}

In [8]:
# 마크다운 (.md) 노드 파서
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core import SimpleDirectoryReader

# 마크다운 파일 로드
markdown_docs = SimpleDirectoryReader(input_dir='./data', required_exts=['.md']).load_data()

parser = MarkdownNodeParser()
markdown_nodes = parser.get_nodes_from_documents(markdown_docs)

In [9]:
len(markdown_nodes)

3

In [10]:
markdown_nodes[0].text

'# 인공지능\r\n인공지능은 세상을 변화시키고 있습니다.'

In [11]:
markdown_nodes[0].metadata

{'file_path': 'c:\\Users\\jangc\\ai_class\\llm_labs\\data\\markdown_example.md',
 'file_name': 'markdown_example.md',
 'file_size': 182,
 'creation_date': '2026-05-28',
 'last_modified_date': '2026-06-03',
 'header_path': '/'}

# 벡터 스토어

- 임베딩을 통해 생성된 고차원 벡터 데이터를 효율적으로 저장하고 검색할 수 있도록 설계된 데이터베이스
- 벡터 간의 거리(유클리드 거리, 코사인 유사도 등)나 유사도를 기반으로 데이터를 검색한다.
- 자연어 처리 (NLP), 추천 시스템, 이미지 검색 등 비정형 데이터를 다루는 ai 앱에서 중요한 역할을 한다.

## FAISS

- 메타에서 만든 고성능 벡터 검색 라이브러리
- 대규모 데이터세트에서도 빠르게유사한 벡터를 검색할 수 있는 도구
- 최근접 이웃 탐색(NNS)에 최적화 되어 있다.
- GPU 가속 기능을 통해 초고속 검색 성능을 제공
- 메모리에서 동작하므로 매우 빠르다.
    - 데이터를 저장하기 않고, 프로그램 종료 시 메모리에 있는 데이터가 사라지므로 별도의 데이터를 저장하거나 복원하는 작업이 필요하다.
- 로컬 환경에서 사용
- 클라우드 기반의 확장성을 지원하지 않는다.

In [13]:
import faiss
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core import StorageContext
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

In [14]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [15]:
# llm 설정
Settings.llm = OpenAI(model='gpt-4o', temperature=0.5, max_tokens=200, context_window=4096)

In [16]:
# 데이터 로드
documents = SimpleDirectoryReader('./data/pdf_sample2').load_data()

In [17]:
# FAISS 인덱스 생성
dimension = 1536 # OpenAI 임베딩 차원
faiss_index = faiss.IndexFlatL2(dimension)

In [18]:
# FAISS를 라마인덱스의 인덱싱 및 검색 파이프라인에 통합
vector_store = FaissVectorStore(faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 문서 내용을 벡터 DB에 저장
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)